In [1]:
import yfinance as yf
import pandas as pd
import talib
import torch
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE



In [2]:
# data = yf.download("^GDAXI", start="2019-01-01", end="2024-01-01")
# data.to_csv('index_stock.csv')

df = pd.read_csv('index_stock.csv')
print(df.shape)
print(df.head())


(1274, 6)
        Price             Close              High               Low  \
0      Ticker            ^GDAXI            ^GDAXI            ^GDAXI   
1        Date               NaN               NaN               NaN   
2  2019-01-02  10580.1904296875  10612.7197265625  10386.9697265625   
3  2019-01-03    10416.66015625    10538.66015625  10400.1103515625   
4  2019-01-04  10767.6904296875    10786.33984375   10483.900390625   

               Open    Volume  
0            ^GDAXI    ^GDAXI  
1               NaN       NaN  
2    10477.76953125  79626700  
3  10467.1103515625  84733800  
4  10533.9404296875  95339500  


In [3]:
df = df.iloc[2:]

In [4]:
df

,Price,Close,High,Low,Open,Volume
2,2019-01-02,10580.1904296875,10612.7197265625,10386.9697265625,10477.76953125,79626700
3,2019-01-03,10416.66015625,10538.66015625,10400.1103515625,10467.1103515625,84733800
4,2019-01-04,10767.6904296875,10786.33984375,10483.900390625,10533.9404296875,95339500
5,2019-01-07,10747.8095703125,10814.4697265625,10681.26953125,10814.3896484375,71151400
6,2019-01-08,10803.98046875,10910.7099609375,10745.0302734375,10750.1904296875,93672200
...,...,...,...,...,...,...
1269,2023-12-21,16687.419921875,16708.349609375,16624.16015625,16667.310546875,57871300
1270,2023-12-22,16706.1796875,16735.3203125,16651.779296875,16673.30078125,46295300
1271,2023-12-27,16742.0703125,16775.7109375,16697.580078125,16727.76953125,37678900
1272,2023-12-28,16701.55078125,16783.7890625,16688.51953125,16780.94921875,36091600


In [5]:
df['Doji'] = talib.CDLDOJI(df['Open'], df['High'], df['Low'], df['Close'])
df['Hammer'] = talib.CDLHAMMER(df['Open'], df['High'], df['Low'], df['Close'])
df['Engulfing'] = talib.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])


In [6]:
def min_max_normalization(x, columns=[]):
    x = x.astype(float)
    x_scaled = (x - x.min()) / (x.max() - x.min())
    return x_scaled


In [7]:
df[['Close', 'High', 'Low', 'Open', 'Volume']] = df[['Close', 'High', 'Low', 'Open', 'Volume']].apply(min_max_normalization)

In [8]:
df = df.rename(columns = {'Price':'Date'})

In [9]:
df = df.reset_index()

In [10]:
df.drop(columns='index')
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0,0,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0,0,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0,0,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0,0,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0,0,0


In [11]:
df.value_counts(['Doji', 'Hammer', 'Engulfing'])

Doji  Hammer  Engulfing
0     0        0           997
100   0        0           177
0     0       -100          45
               100          28
      100      0            22
100   100      0             3
Name: count, dtype: int64

In [12]:
def define_pattern(x):
    if x['Doji'] == 0 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 0
    elif x['Doji'] == 100 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 1
    elif x['Doji'] == 0 and x['Hammer'] == 100 and x['Engulfing'] == 0:
        return 2
    elif x['Doji'] == 0 and x['Hammer'] == 0 and (x['Engulfing'] != 0):
        return 3
    else:
        return -1 
    


In [13]:
df['Pattern'] = df.apply(define_pattern, axis=1)

In [14]:
df

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1267,1269,2023-12-21,0.987189,0.964615,0.984013,0.966700,0.146646,0,0,0,0
1268,1270,2023-12-22,0.989435,0.967851,0.987261,0.967409,0.116946,0,0,0,0
1269,1271,2023-12-27,0.993731,0.972697,0.992646,0.973853,0.094839,0,0,0,0
1270,1272,2023-12-28,0.988880,0.973666,0.991581,0.980144,0.090766,0,0,-100,3


In [15]:
df['Pattern'].value_counts()

Pattern
 0    997
 1    177
 3     73
 2     22
-1      3
Name: count, dtype: int64

In [16]:
df = df[df['Pattern'] != -1]
df = df.drop(columns =['Doji', 'Hammer', 'Engulfing'])

In [17]:
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0


## Building neural network with Torch

In [18]:
W1 = torch.randn(5,8,requires_grad = True)
b1 = torch.zeros(8,requires_grad = True)
W2 = torch.randn(8,4,requires_grad = True)
b2 = torch.zeros(4,requires_grad = True)

In [19]:
def forward_pass(X):
    hidden_lay_pre_act = torch.matmul(X, W1) + b1
    relu_act = torch.relu(hidden_lay_pre_act)
    logits = torch.matmul(relu_act, W2) + b2
    return logits

In [20]:
freq = df['Pattern'].value_counts().sort_index()
weights = 1/freq

In [21]:
tensor_1 = torch.tensor(weights.values, dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight = tensor_1)

In [22]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)

In [23]:
for epoch in range(100):
    logits = forward_pass(torch.tensor(df[['Close', 'High', 'Low', 'Open', 'Volume']].values, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(df['Pattern'].values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

2.6609649658203125
1.4658430814743042
1.4954074621200562
1.4032630920410156
1.4067370891571045
1.3974308967590332
1.3914986848831177
1.3896435499191284
1.3873494863510132
1.3855475187301636


## Split data

In [24]:
df_train, df_test = train_test_split(df, test_size = 0.2)

In [25]:
X_train = df_train[['Close', 'High', 'Low', 'Open', 'Volume']]
X_test = df_test[['Close', 'High', 'Low', 'Open', 'Volume']]
y_train = df_train['Pattern']
y_test = df_test['Pattern']


In [26]:
for epoch in range(500):
    logits = forward_pass(torch.tensor(X_train.values, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(y_train.values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

1.3901801109313965
1.3802719116210938
1.3776829242706299
1.3753918409347534
1.3736488819122314
1.3721275329589844
1.3707382678985596
1.3695012331008911
1.3683843612670898
1.3673685789108276
1.3664056062698364
1.3654747009277344
1.3645716905593872
1.36368989944458
1.3628326654434204
1.3619986772537231
1.3611972332000732
1.3604516983032227
1.3597383499145508
1.3590530157089233
1.3583931922912598
1.3577524423599243
1.3571349382400513
1.3565367460250854
1.3559626340866089
1.355403184890747
1.3548943996429443
1.3544187545776367
1.3539633750915527
1.3535254001617432
1.3531038761138916
1.352697730064392
1.3523058891296387
1.3519282341003418
1.3515633344650269
1.3512111902236938
1.3508727550506592
1.350545048713684
1.3502275943756104
1.3499188423156738
1.3496174812316895
1.3493261337280273
1.3490411043167114
1.3487634658813477
1.3484915494918823
1.34822678565979
1.3479658365249634
1.3477131128311157
1.3474634885787964
1.3472203016281128


In [27]:
logits_test = forward_pass(torch.tensor(X_test.values, dtype=torch.float32))
logits_test

tensor([[ 0.0092,  0.1107, -0.6328,  0.1193],
        [-0.2977, -0.3324, -0.3660, -0.6160],
        [-0.2984, -0.3658, -0.3463, -0.6350],
        ...,
        [-0.2545, -0.2399, -0.4317, -0.5343],
        [-0.1416,  0.2485, -0.6974,  0.0235],
        [ 0.1077,  0.1109, -0.6194,  0.3864]], grad_fn=<AddBackward0>)

In [28]:
test_result = torch.argmax(logits_test, dim=1)
test_result

tensor([3, 0, 0, 1, 3, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 3, 3, 1, 0, 1, 1, 3,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 3, 1, 1, 1, 1, 1, 3, 1,
        2, 0, 1, 1, 1, 3, 3, 3, 1, 3, 1, 0, 0, 1, 3, 1, 3, 3, 0, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 2, 1, 0, 3, 3, 1, 2, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 3, 3, 1, 1, 1, 1, 0, 1, 1, 0, 3, 3, 1, 0, 1, 1, 1,
        1, 1, 0, 3, 3, 0, 1, 1, 1, 1, 3, 1, 0, 2, 3, 3, 0, 1, 1, 1, 1, 1, 1, 0,
        1, 1, 3, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 3, 1, 1, 3, 1,
        1, 0, 1, 0, 1, 0, 0, 3, 1, 1, 1, 1, 0, 3, 0, 2, 0, 1, 1, 1, 1, 0, 0, 0,
        3, 1, 1, 1, 2, 1, 1, 1, 0, 1, 0, 3, 0, 1, 0, 3, 1, 1, 3, 3, 3, 2, 3, 1,
        1, 0, 0, 1, 0, 0, 1, 0, 3, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 3, 1, 1,
        1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 2, 1, 1, 3])

In [29]:
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.76      0.27      0.40       195
           1       0.13      0.58      0.21        31
           2       0.00      0.00      0.00        10
           3       0.10      0.22      0.14        18

    accuracy                           0.29       254
   macro avg       0.25      0.27      0.19       254
weighted avg       0.61      0.29      0.34       254



In [30]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [31]:
for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

1.3683979511260986
1.2354280948638916
1.3413268327713013
1.187791347503662
1.4376529455184937
1.2756632566452026
1.5361380577087402
1.3462456464767456
1.5045526027679443
1.3405340909957886
1.4318156242370605
1.5218842029571533
1.264201283454895
1.6812466382980347
1.4126776456832886
1.3182190656661987
1.2217377424240112
1.598706603050232
1.1765296459197998
1.2182198762893677
1.2267177104949951
1.1418014764785767
1.1102255582809448
1.308485746383667
1.1208410263061523
1.1444456577301025
2.3586888313293457
1.9335418939590454
1.1118903160095215
1.2206823825836182
1.2584141492843628
1.2579829692840576
1.4994057416915894
1.2152514457702637
1.3778671026229858
1.512357473373413
1.0858149528503418
1.2391334772109985
1.187421202659607
1.1717884540557861
1.2209758758544922
1.417731523513794
1.435203194618225
1.124928593635559
1.1459952592849731
1.2412678003311157
1.3177968263626099
1.5665514469146729
1.4605919122695923
1.281632661819458
1.3232190608978271
1.2374179363250732
1.512174367904663
1.31

In [32]:
logits_test_batch = forward_pass(torch.tensor(X_test.values, dtype=torch.float32))
test_result = torch.argmax(logits_test_batch, dim=1)
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.77      0.27      0.40       195
           1       0.14      0.61      0.23        31
           2       0.00      0.00      0.00        10
           3       0.07      0.17      0.10        18

    accuracy                           0.30       254
   macro avg       0.24      0.26      0.18       254
weighted avg       0.61      0.30      0.34       254



## Applying imbalanced learn

In [33]:
sm = SMOTE()
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

In [36]:
y_train_sm.value_counts()

Pattern
3    802
0    802
1    802
2    802
Name: count, dtype: int64

In [35]:
X_train

,Close,High,Low,Open,Volume
187,0.460548,0.437283,0.466536,0.441488,0.200658
1007,0.652645,0.641021,0.653726,0.648598,0.431271
143,0.476275,0.451069,0.482158,0.457588,0.180609
633,0.846391,0.837381,0.847908,0.842217,0.136096
963,0.503707,0.485714,0.487085,0.469853,0.144912
...,...,...,...,...,...
752,0.813862,0.795172,0.800103,0.787500,0.175503
1227,0.752969,0.735102,0.752478,0.733514,0.310674
129,0.491074,0.470193,0.499981,0.478565,0.231859
557,0.736874,0.711853,0.738912,0.718024,0.000116
